# Análisis descriptivo: generación de tabla de contingencia para prueba de independencia para variables categóricas

En este Notebook se tiene como objetivo construir una tabla de contingencia donde a lo largo de las filas se dispongan los valores diferentes del campo seleccionado del conjunto de **variables fijas**, y a lo largo de las columnas, se ubicarán las variables *day_of_the_week*, *time_slot* y *converted* en el orden mencionado. Se muestra la tabla y a continuación se construye un script que automatiza la ejecución de una prueba de bondad de ajuste sobre cada tabla creada para filtrar las variables explicativas de las que se servirán los modelos de Machine Learning para entrenarse. Los elementos del conjunto de *variables fijas* se listan a continuación sin su descripción que ya ha sido abarcada en el documento TFM:

 - **operator_name**
 - **strategy_dial**
 - **genre**
 - **age** (Se construye la variable age_range para obtener la edad categórica)
 - **city**
 - **us_congress**
 - **st_senate**
 - **house_district**
 - **sboe**
 - **party**
 - **ethnic_description**

El conjunto de **variables libres** no se somete naturalmente a un proceso de selección de variables explicativas: 
 - **day_of_the_week**
 - **time_slot**

La variable **state** no se incluye porque es una jerarquía superior al estado. Son variables dependientes y su inclusión implicaría redundancia de información. Enseguida empieza el script

In [1]:
# Importación de librerias necesarias
from deltalake import DeltaTable
import pandas as pd
import warnings

# Configuraciones adicionales para mostrar tablas
pd.set_option("display.max_colwidth", None)-
pd.set_option("display.max_columns", None)
warnings.filterwarnings("ignore")

#### ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
## Lectura de tabla delta de llamadadas y conversión a Dataframe de Pandas para facilitar manejo


#### ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [2]:
# Path asociado a la capa Silver de llamadas completamente poblada 
path_dt_calls = r"C:\Users\TPG\Documents\Trabajos UNIR\Semestre II\TFM\Data Call Center All\Silver\Calls"
df_calls = DeltaTable(path_dt_calls).to_pandas()
df_calls.sample(5)

,call_date,day_of_the_week,time_slot,operator_name,list_id,gmt_offset_now,voter_id,length_in_sec,strategy_dial,candidate_name,initial_call_status_name,final_call_status_name,yard_sign,top_issue,campaign,converted
93848,2025-02-21 17:56:00+00:00,Friday,noon,Agent 4,104,-5.0,ad23b16e08d6f3523a91132ae23e144742fdd7399df50da41f595fdbeba98bf7,13,Main,candidate_name_2,Answering Machine,NULL,NULL,NULL,OLD,No
399446,2024-11-05 23:09:00+00:00,Tuesday,evening,Outbound Auto Dial,366,-8.0,15164709605eb78d85f1074edeec970581686448205ef9ec79e3660ea3e43dcf,0,None,candidate_name_3,Answering Machine Auto,NULL,NULL,NULL,NULL,No
110820,2025-02-22 22:18:00+00:00,Saturday,afternoon,Outbound Auto Dial,104,-5.0,411043ebd1eb63c57f87b369489438169557a1e337228bb4c1ee5265f7e282a6,0,None,candidate_name_2,Answering Machine Auto,NULL,NULL,NULL,OLD,No
284439,2025-03-04 21:17:00+00:00,Tuesday,afternoon,Outbound Auto Dial,104,-5.0,843ef35186f5ecadc0b86a9c5e2fe78270ce6c463473cdb230c27cea64f41555,0,None,candidate_name_2,Answering Machine Auto,NULL,NULL,NULL,GOTV,No
363022,2024-11-02 17:05:00+00:00,Saturday,noon,Outbound Auto Dial,363,-8.0,6b56b68f441a6eca1bb6d0c838d99a2eedb89aae2b8d9466811fe4fc3b5db9d0,50,None,candidate_name_3,Answering Machine Msg Played,NULL,NULL,NULL,NULL,No


In [29]:
filtro = df_calls["voter_id"]=="6e5cc6bcd9e1fc8c70fa041735d98cd89312602693473e094c17e804dbc8e6ea"
df_calls[filtro]

,call_date,day_of_the_week,time_slot,operator_name,list_id,gmt_offset_now,voter_id,length_in_sec,strategy_dial,candidate_name,initial_call_status_name,final_call_status_name,yard_sign,top_issue,campaign,converted
303331,2024-10-31 02:21:00+00:00,Wednesday,evening,Outbound Auto Dial,363,-7.0,6e5cc6bcd9e1fc8c70fa041735d98cd89312602693473e094c17e804dbc8e6ea,0,None,candidate_name_3,Disconnected Number Auto,NULL,NULL,NULL,NULL,No


In [3]:
# Obtención de metadata elemental del dataframe
df_calls.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 436624 entries, 0 to 436623
Data columns (total 16 columns):
 #   Column                    Non-Null Count   Dtype              
---  ------                    --------------   -----              
 0   call_date                 436624 non-null  datetime64[us, UTC]
 1   day_of_the_week           436624 non-null  object             
 2   time_slot                 436624 non-null  object             
 3   operator_name             436624 non-null  object             
 4   list_id                   436624 non-null  object             
 5   gmt_offset_now            436624 non-null  float64            
 6   voter_id                  436624 non-null  object             
 7   length_in_sec             436624 non-null  int32              
 8   strategy_dial             436624 non-null  object             
 9   candidate_name            436624 non-null  object             
 10  initial_call_status_name  436624 non-null  object             
 11  

In [28]:
# Revisemos el desbalance de las clases de converted
df_calls["converted"].value_counts(normalize=True)

converted
No     0.965158
Yes    0.034842
Name: proportion, dtype: float64

In [4]:
# Se crea el dataframe de llamadas a combinar únicamente con las variables fijas y libres involucradas. Con la variable de relación y el campo converted incluido por supuesto.
filtro_cols_llamadas = ["voter_id", "day_of_the_week", "time_slot", "operator_name", "strategy_dial", "converted", "candidate_name"]
df_calls_combinar = df_calls[filtro_cols_llamadas]

#### ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
## Lectura de tabla delta de votantes y conversión a Dataframe de Pandas para facilitar manejo
#### ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [5]:
# Path asociado a la capa Silver de llamadas completamente poblada 
path_dt_voters = r"C:\Users\TPG\Documents\Trabajos UNIR\Semestre II\TFM\Data Call Center All\Silver\Voters"
df_voters = DeltaTable(path_dt_voters).to_pandas()

# Generación de columna age_range
# Definimos los intervalos (bins) y sus etiquetas
bins = [17, 34, 54, 74, 120]
labels = ['YoungAdult','Adult','Senior','Elderly']

# Generamos la serie categórica
df_voters["age_range"] = pd.cut(df_voters["age"], bins=bins, labels=labels, right=True)

# Se muestran 5 registros aleatoriamente del dataframe
df_voters.sample(5)

,voter_id,genre,age,us_congress,city,state,st_senate,house_district,sboe,party,ethnic_description,age_range
230458,30e3488df43b2f5c128871e8d28c5a899189925237ec511808852d7b7c660ccb,M,27,CD-AVK,City-EVK,State-VK,ST-CVK,HD-BVK,NULL,Party-AVK,Ethnic-AVK,YoungAdult
80425,55bf3bcf2e6e1c04b1eb2f62571aae6e6f4fa7e35cdfff3b669dd99fde159d01,F,22,CD-AVK,City-EVK,State-VK,ST-CVK,HD-BVK,NULL,Party-AVK,Ethnic-CVK,YoungAdult
120047,9091aeb2f5bae6fb3a4ee86ae3e99b3bc345406fc502c20b4909b879c1db4815,F,41,CD-AVK,City-DVK,State-VK,ST-CVK,HD-AVK,NULL,Party-AVK,Ethnic-EVK,Adult
303760,0934e39ee59cd23808b9d373aed56f7770d4855b7795c5f45b6c047072428ac9,M,76,CD-BRB,City-ERB,State-RB,ST-DRB,HD-DRB,NULL,NULL,NULL,Elderly
31038,c081215ee1931fd1ca514d30eb03aa9083092aafe3d3d01ce5c7eb0183806159,M,78,CD-ARB,City-NRB,State-RB,ST-BRB,HD-CRB,NULL,NULL,NULL,Elderly


In [6]:
# Obtención de metadata elemental del dataframe
df_voters.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 318659 entries, 0 to 318658
Data columns (total 12 columns):
 #   Column              Non-Null Count   Dtype   
---  ------              --------------   -----   
 0   voter_id            318659 non-null  object  
 1   genre               318659 non-null  object  
 2   age                 318659 non-null  int32   
 3   us_congress         318659 non-null  object  
 4   city                318659 non-null  object  
 5   state               318659 non-null  object  
 6   st_senate           318659 non-null  object  
 7   house_district      318659 non-null  object  
 8   sboe                318659 non-null  object  
 9   party               318659 non-null  object  
 10  ethnic_description  318659 non-null  object  
 11  age_range           318658 non-null  category
dtypes: category(1), int32(1), object(10)
memory usage: 25.8+ MB


In [7]:
# Se crea el dataframe de votantes a combinar únicamente con las variables fijas involucradas y la clave de relación voter_id.
#filtro_cols_llamadas = ["genre", "age_range", "us_congress", "city", "st_senate", "house_district", "sboe", "party", "ethnic_description", "age_range"]
filtro_cols_votantes = ["voter_id", "genre", "age_range", "us_congress", "city", "st_senate", "house_district", "party", "ethnic_description", "sboe"]
df_voters_combinar = df_voters[filtro_cols_votantes]

#### ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
## Obtención de dataframe a agrupar y pivotear

#### ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Para conseguir el dataframe completo del que se generará la tabla de contingencia, se realizará un combinación left a partir con clave de relación voter_id 


In [8]:
df_calls_voters = pd.merge(df_calls_combinar, df_voters_combinar, on="voter_id", how="left")
df_calls_voters.sample(5)

,voter_id,day_of_the_week,time_slot,operator_name,strategy_dial,converted,candidate_name,genre,age_range,us_congress,city,st_senate,house_district,party,ethnic_description,sboe
28848,63804ccbae08e07609bc37d8a1f95abfb5e1d374f727b2b6ea9d9a5de6c0cf96,Thursday,evening,Agent 3,Main,No,candidate_name_2,F,Adult,CD-ARB,City-HRB,ST-DRB,HD-DRB,NULL,NULL,NULL
179132,3e47b8697875e573556bd64e433b3d317efae851680bfb1d8d031737d4b36531,Thursday,evening,Outbound Auto Dial,None,No,candidate_name_2,M,Senior,CD-ARB,City-CRB,ST-CRB,HD-BRB,NULL,NULL,NULL
80991,a0bcccc811f067e97d07f9661f712402c3c280f79d430378f0ec7d7ff17e8445,Thursday,afternoon,Agent 1,Main,No,candidate_name_2,F,Adult,CD-ARB,City-KRB,ST-FRB,HD-FRB,NULL,NULL,NULL
188788,94e6de1bc29347bab42806a2037031a5dc6319c67594b85383f7d594c8dfa54e,Sunday,afternoon,Outbound Auto Dial,None,No,candidate_name_2,F,Senior,CD-BRB,City-HRB,ST-ERB,HD-ARB,NULL,NULL,NULL
393854,be74a0deb2eb7d96732c916901d58191386a34672b061dc397411dee0ddd049b,Monday,evening,Outbound Auto Dial,None,No,candidate_name_3,F,Adult,CD-AVK,City-EVK,ST-CVK,HD-BVK,Party-CVK,Ethnic-BVK,NULL


In [30]:
filtro = (df_calls_voters["voter_id"]=="6e5cc6bcd9e1fc8c70fa041735d98cd89312602693473e094c17e804dbc8e6ea")
df_calls_voters[filtro]

,voter_id,day_of_the_week,time_slot,operator_name,strategy_dial,converted,candidate_name,genre,age_range,us_congress,city,st_senate,house_district,party,ethnic_description,sboe
303388,6e5cc6bcd9e1fc8c70fa041735d98cd89312602693473e094c17e804dbc8e6ea,Wednesday,evening,Outbound Auto Dial,None,No,candidate_name_3,M,YoungAdult,CD-AVK,City-IVK,ST-CVK,HD-AVK,Party-BVK,Ethnic-EVK,NULL


In [9]:
# Conteo de llamadas por operador
vc = df_calls_voters['operator_name'].value_counts()
print(vc.head(15))   # ver top 50
print("Total operarios:", vc.shape[0])

operator_name
Outbound Auto Dial    371657
Agent 4                12782
Agent 1                11988
Agent 2                 8519
Agent 3                 8222
Agent 5                 5752
Agent 10                2215
Agent 19                1823
Agent 8                 1757
Agent 16                1570
Agent 15                1506
Agent 7                 1489
Agent 14                1102
Agent 20                 911
Agent 6                  889
Name: count, dtype: int64
Total operarios: 26


In [10]:
# Conteo de llamadas por estrategia de marcado
vc_strategy = df_calls_voters['strategy_dial'].value_counts()
print(vc_strategy.head(10))   # ver top 50
print("Total estrategias:", vc_strategy.shape[0])

strategy_dial
None    371861
Main     63991
60         126
11          56
10          54
9           51
1           43
8           42
12          40
0           36
Name: count, dtype: int64
Total estrategias: 62


In [11]:
# Conteo de llamadas por ciudad
vc_city = df_calls_voters['city'].value_counts()
print(vc_city.head(10))   # ver top 50
print("Total de ciudades:", vc_city.shape[0])

city
City-JRB    39858
City-DVK    35538
City-EVK    28178
City-HRB    28010
City-BRB    19853
City-BVK    16103
City-TRB    15440
City-IVK    13881
City-HVK    13572
City-ERB    12481
Name: count, dtype: int64
Total de ciudades: 71


In [12]:
# Conteo de llamadas por distrito congresional
vc_congress = df_calls_voters['us_congress'].value_counts()
print(vc_congress.head(10))   # ver top 50
print("Total de distritos congresionales:", vc_congress.shape[0])

us_congress
CD-ARB    141611
CD-AVK    134049
CD-BRB     91125
CD-BMH      4679
CD-AMH      2182
CD-CMH       411
CD-CRB         9
Name: count, dtype: int64
Total de distritos congresionales: 7


In [13]:
# Conteo de llamadas por partido políticos
vc_party = df_calls_voters['party'].value_counts()
print(vc_party.head(10))   # ver top 50
print("Total de partidos políticos:", vc_party.shape[0])

party
NULL         240022
Party-AVK     64069
Party-CVK     40994
Party-BVK     20638
Party-FVK      4512
Party-DVK      1306
Party-EVK      1028
Party-GVK       721
Party-HVK       670
Party-IVK        73
Name: count, dtype: int64
Total de partidos políticos: 12


In [14]:
# Conteo de llamadas por partido políticos
vc_ethnic = df_calls_voters['ethnic_description'].value_counts()
print(vc_ethnic.head(15))   # ver top 50
print("Total de grupos étnicos:", vc_ethnic.shape[0])

ethnic_description
NULL          253088
Ethnic-AVK     38249
Ethnic-BVK     36489
Ethnic-EVK     32548
Ethnic-DVK     10037
Ethnic-CVK      3655
Name: count, dtype: int64
Total de grupos étnicos: 6


In [15]:
# Conteo de llamadas por partido políticos
vc_senate = df_calls_voters['st_senate'].value_counts()
print(vc_senate.head(15))   # ver top 50
print("Total de distritos senatoriales:", vc_senate.shape[0])

st_senate
ST-CVK    73739
ST-ERB    46722
ST-CRB    42178
ST-BRB    40537
ST-DRB    37852
ST-BVK    32781
ST-AVK    27527
ST-GRB    21657
ST-FRB    15638
ST-ARB    15209
ST-HRB    12952
ST-BMH     5960
ST-AMH     1312
NULL          2
Name: count, dtype: int64
Total de distritos senatoriales: 14


In [16]:
# Conteo de llamadas por distrito de representante
vc_house = df_calls_voters['house_district'].value_counts()
print(vc_house.head(15))  
print("Total de distritos congresionales:", vc_house.shape[0])

house_district
HD-AVK    78936
HD-ARB    46722
HD-BRB    42178
HD-CRB    40537
HD-DRB    37852
HD-BVK    28389
HD-CVK    26722
HD-ERB    21657
HD-FRB    15638
HD-GRB    12222
HD-JRB     7829
HD-BMH     5728
HD-IRB     5123
HD-HRB     2987
HD-AMH     1362
Name: count, dtype: int64
Total de distritos congresionales: 18


In [17]:
# Conteo de llamadas por sboe
vc_sboe = df_calls_voters['sboe'].value_counts()
print(vc_sboe.head(15))  
print("Total de distritos sboe:", vc_sboe.shape[0])

sboe
NULL        366794
SBOE-BMH      3536
SBOE-CMH      2976
SBOE-AMH       760
Name: count, dtype: int64
Total de distritos sboe: 4


#### Agrupamiento de categorías minoritaras

En el entrenamiento de modelos de ML e incluso en la selección previa de variables potencialmente explicativas resulta de gran utilidad agrupar categorías minoritaras. Se consigue menor ruido en el entrenamiento y métricas más interpretables

In [18]:
# Ante la presencia de operadores con tan pocas llamadas que se reducen notablemente entre las combinaciones de
# día de llamada, franja horaria, y valor de converted, se agrupan los agentes como menos de 1000 llamadas en la categoría "Minority Operators"
min_count = 1000  # ejemplo: agrupar operadores con < 30 llamadas
min_count_city = 2000 # Cuando se distribuyen las llamadas por ciudad a lo largo de día de la semana y franja horaria, se verifican valores muy pequeños de llamadas lanzadas
                      # en las ciudades minoritarias 
# identificar operadores minoritarios
minority_ops = vc[vc < min_count].index.tolist()

# identificar estrategias minoritarias
minority_strategy = vc_strategy[vc_strategy < min_count].index.tolist()

# Identificar ciudades minoritarias
minority_city = vc_city[vc_city < min_count_city].index.tolist()

# Identificar distritos de congreso minoritarias
minority_congress = vc_congress[vc_congress < min_count].index.tolist()

# Identificar partidos politicos minoritarias
minority_party = vc_party[vc_party < min_count].index.tolist()

# 
df_calls_voters['operator_name'] = df_calls_voters['operator_name'].where(
    ~df_calls_voters['operator_name'].isin(minority_ops),
    other='Minority Operators'
)

# 
df_calls_voters['strategy_dial'] = df_calls_voters['strategy_dial'].where(
    ~df_calls_voters['strategy_dial'].isin(minority_strategy),
    other='Minority Strategy'
)

# 
df_calls_voters['city'] = df_calls_voters['city'].where(
    ~df_calls_voters['city'].isin(minority_city),
    other='Minority City'
)

# 
df_calls_voters['us_congress'] = df_calls_voters['us_congress'].where(
    ~df_calls_voters['us_congress'].isin(minority_congress),
    other='Minority Congress'
)

# 
df_calls_voters['party'] = df_calls_voters['party'].where(
    ~df_calls_voters['party'].isin(minority_party),
    other='Minority Party'
)

# ver cuántas filas fueron agrupadas para operadores
n_grouped = df_calls_voters['operator_name'].eq('Minority Operators').sum()
print(f"Operadores agrupados: {len(minority_ops)}, filas agrupadas: {n_grouped}")

# ver cuántas filas fueron agrupadas para estrategias
n_grouped_str = df_calls_voters['strategy_dial'].eq('Minority Strategy').sum()
print(f"Estrategias agrupadas: {len(minority_strategy)}, filas agrupadas: {n_grouped_str}")

# ver cuántas filas fueron agrupadas para las ciudades minoritarias
n_grouped_city = df_calls_voters['city'].eq('Minority City').sum()
print(f"Ciudades agrupadas: {len(minority_city)}, filas agrupadas: {n_grouped_city}")

# ver cuántas filas fueron agrupadas para los distritos congresionales
n_grouped_congress = df_calls_voters['us_congress'].eq('Minority Congress').sum()
print(f"Distritos congresionales agrupados: {len(minority_congress)}, filas agrupadas: {n_grouped_congress}")

# ver cuántas filas fueron agrupadas para los partidos politicos
n_grouped_party = df_calls_voters['party'].eq('Minority Party').sum()
print(f"Partidos políticos agrupados: {len(minority_party)}, filas agrupadas: {n_grouped_party}")

Operadores agrupados: 13, filas agrupadas: 6308
Estrategias agrupadas: 60, filas agrupadas: 838
Ciudades agrupadas: 38, filas agrupadas: 25908
Distritos congresionales agrupados: 2, filas agrupadas: 420
Partidos políticos agrupados: 5, filas agrupadas: 1497


In [19]:
df_calls_voters["party"].value_counts()

party
NULL              240022
Party-AVK          64069
Party-CVK          40994
Party-BVK          20638
Party-FVK           4512
Minority Party      1497
Party-DVK           1306
Party-EVK           1028
Name: count, dtype: int64

#### ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
## Obtención de tabla de contingencia y resultados de prueba de bondad de ajuste para cada campo en el conjunto de potenciales variables fijas

#### ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

#### Creación de matriz cruzada dinámica de la tasa de conversión **TC**

¡Agregar en el cuadro de mando con perspectiva interna!

In [32]:
import ipywidgets as widgets
from IPython.display import display
import numpy as np

# 1) Normalizar columna 'converted' a bandera 0/1
def to_flag_conv(x):
    if pd.isna(x):
        return np.nan
    # números (0/1)
    if isinstance(x, (int, np.integer, float, np.floating)):
        if x == 1: return 1
        if x == 0: return 0
        # otros num -> NaN
        return np.nan
    s = str(x).strip().lower()
    if s in {'1','true','t','yes','y','si','sí','s'}:
        return 1
    if s in {'0','false','f','no','n'}:
        return 0
    return np.nan

df = df_calls_voters.copy()  # usa tu df real

# Filtrar df para verificar estrategias de modelo
filtro = (df["age_range"]=="YoungAdult") & (df["city"]=="City-IVK")
df = df[filtro]

df['converted_flag'] = df['converted'].apply(to_flag_conv)

# Definición de variables fijas
FIXED_VARS = [
    'operator_name', 'strategy_dial', 'genre', 'age_range', 'city',
    'us_congress', 'st_senate', 'house_district', 'sboe', 'party', 'ethnic_description',
    'candidate_name'
]

def create_pivot(fixed_field):
    pivot_table = (
        df
        .groupby(["day_of_the_week", "time_slot", fixed_field])["converted_flag"]
        .mean()
        .unstack(fill_value=0)
        * 100
    )
    display(pivot_table.round(2))

# Dropdown con los campos disponibles
fixed_field_dropdown = widgets.Dropdown(
    options=FIXED_VARS,  # ← aquí agregas los campos fijos que quieras
    value="genre",
    description="Field:"
)

widgets.interact(create_pivot, fixed_field=fixed_field_dropdown)

interactive(children=(Dropdown(description='Field:', index=2, options=('operator_name', 'strategy_dial', 'genr…

<function __main__.create_pivot(fixed_field)>

In [21]:
pivot_table_test = (
        df
        .groupby(["candidate_name"])["converted_flag"]
        .mean()
        #.unstack(fill_value=0)
        * 100
    )
display(pivot_table_test.round(2))

candidate_name
candidate_name_1    3.02
candidate_name_2    3.99
candidate_name_3    2.42
Name: converted_flag, dtype: float64

In [22]:
from scipy.stats import chi2_contingency
import numpy as np

tab = pd.crosstab(df['converted'], df['candidate_name'])
chi2, p, dof, exp = chi2_contingency(tab.values, correction=False)
n = tab.values.sum()
r,k = tab.shape
cramers_v = np.sqrt(chi2 / (n * min(r-1, k-1)))
print("chi2:",chi2,"p:",p,"cramers_v:",cramers_v)
tab

chi2: 688.48165330968 p: 3.1485350912361425e-150 cramers_v: 0.039706313057171476


candidate_name,candidate_name_1,candidate_name_2,candidate_name_3
converted,,,
No,7044,281820,132612
Yes,219,11711,3284


In [23]:
# Prueba para party
#filtro = (df_calls_voters["sboe"]!="NULL") & (df_calls_voters["candidate_name"]=="candidate_name_2")
#df_calls_voters = df_calls_voters[filtro]

In [24]:
# Código de ayuda para construir la tabla de contingencia, mostrarla con un "filtro" (selector)
# y ejecutar automáticamente pruebas de bondad de ajuste (chi2) sobre cada tabla.
# - Comprueba si existe `df_calls_voters`. Si no existe, crea un dataset demo pequeño para mostrar el flujo.
# - Funciones principales:
#     build_contingency_table(df, fixed_var)
#     run_chi2_tests_on_table(table) -> devuelve dict con chi2, p, dof, cramers_v y warnings
#     run_tests_for_fixed_vars(df, fixed_vars_list)
#     show_interactive_selector(fixed_vars_list) -> si ipywidgets está instalado crea un dropdown interactivo.
# Recomendación de interpretación: p < alpha => hay asociación (útil como variable explicativa). p >= alpha => no evidencia de asociación.
# Ejecuta esto en tu notebook; si ya tienes df_calls_voters, el código lo usará directamente.

import numpy as np
from math import sqrt
from scipy.stats import chi2_contingency
import warnings

# helper display (si el entorno tiene caas_jupyter_tools lo usamos para una tabla interactiva)
try:
    from caas_jupyter_tools import display_dataframe_to_user
    _HAS_DISPLAY_TOOL = True
except Exception:
    _HAS_DISPLAY_TOOL = False


#FIXED_VARS = [
 #   'operator_name', 'strategy_dial', 'genre', 'age_range', 'city',
  #  'us_congress', 'st_senate', 'house_district', 'sboe', 'party', 'ethnic_description'
#]
#FIXED_VARS = [
 #   'operator_name', 'strategy_dial', 'genre', 'age_range', 'city',
  #  'us_congress', 'st_senate', 'house_district'
#]
FIXED_VARS = ["genre", "us_congress", "operator_name", "strategy_dial", "age_range", "city", "party", "ethnic_description", "st_senate", "house_district", "sboe", "candidate_name"]

# --- Construcción de la tabla de contingencia ---
def build_contingency_table(df, fixed_var):
    """
    Construye tabla de contingencia con índice jerárquico: day_of_the_week, time_slot, converted
    columnas: valores únicos de fixed_var
    devuelve un DataFrame con fill_value=0
    """
    if fixed_var not in df.columns:
        raise ValueError(f"La columna '{fixed_var}' no existe en el DataFrame.")
    idx = ['day_of_the_week', 'time_slot', 'converted']
    # aseguramos que las columnas existan
    for c in idx:
        if c not in df.columns:
            raise ValueError(f"La columna requerida '{c}' no está en el DataFrame.")
    # agregamos conteo
    ct = df.groupby(idx + [fixed_var]).size().unstack(fill_value=0)
    # Si quieres una tabla "plana" con rows como combinaciones en una sola columna, puedes hacer:
    ct_flat = ct.copy()
    ct_flat.index = ct_flat.index.map(lambda t: f"{t[0]} | {t[1]} | conv={t[2]}")
    return ct, ct_flat  # retornamos la tabla con índice jerárquico y una versión "plana" para mostrar


# --- Prueba chi2 + Cramer's V ---
def cramers_v(chi2, n, r, k):
    """Calcula el índice de asociación Cramér's V"""
    denom = n * min(r-1, k-1)
    if denom == 0:
        return np.nan
    return sqrt(chi2 / denom)

def run_chi2_tests_on_table(table):
    """
    table: DataFrame 2D (rows x cols) con frecuencias observadas.
    Devuelve dict con chi2, p, dof, expected, cramers_v y avisos.
    """
    result = {}
    # convert to numpy array
    obs = table.values.astype(int)
    n = obs.sum()
    r, k = obs.shape
    # check trivial cases
    if n == 0 or r < 2 or k < 2:
        result.update({'chi2': np.nan, 'p': np.nan, 'dof': np.nan,
                       'cramers_v': np.nan, 'warning': 'Tabla demasiado pequeña o vacía para chi2.'})
        return result
    # Ejecutamos chi2_contingency (scipy)
    try:
        chi2, p, dof, expected = chi2_contingency(obs, correction=False)
    except Exception as e:
        result.update({'chi2': np.nan, 'p': np.nan, 'dof': np.nan,
                       'cramers_v': np.nan, 'warning': f'Error en chi2_contingency: {e}'})
        return result
    # comprobación de frecuencias esperadas pequeñas
    small_expected = (expected < 5).sum()
    warning = ''
    if small_expected > 0:
        warning = (f"{small_expected} celdas tienen frecuencia esperada < 5. Resultado del chi2 puede no ser fiable."
                   " Considera agrupar categorías o usar pruebas exactas/montecarlo para tablas pequeñas.")
    cv = cramers_v(chi2, n, r, k)
    result.update({'chi2': chi2, 'p': p, 'dof': dof, 'expected_min': expected.min(),
                   'expected_mean': expected.mean(), 'cramers_v': cv, 'warning': warning})
    return result


# --- Wrapper que corre pruebas para una lista de fixed_vars ---
def run_tests_for_fixed_vars(df, fixed_vars_list=None, alpha=0.05, verbose=True):
    if fixed_vars_list is None:
        fixed_vars_list = FIXED_VARS
    outcomes = []
    for v in fixed_vars_list:
        if v not in df.columns:
            outcomes.append({'fixed_var': v, 'error': 'no existe en df'})
            continue
        try:
            table_hier, table_flat = build_contingency_table(df, v)
        except Exception as e:
            outcomes.append({'fixed_var': v, 'error': str(e)})
            continue
        # chi2 sobre la tabla completa (filas = combinaciones de day/time/converted)
        res_full = run_chi2_tests_on_table(table_flat)
        # chi2 sobre cada variable individual en las filas frente a fixed_var (por si quieres ver cada caso)
        # day_of_the_week vs fixed_var
        tab_day = pd.crosstab(df['day_of_the_week'], df[v])
        tab_time = pd.crosstab(df['time_slot'], df[v])
        tab_conv = pd.crosstab(df['converted'], df[v])
        res_day = run_chi2_tests_on_table(tab_day)
        res_time = run_chi2_tests_on_table(tab_time)
        res_conv = run_chi2_tests_on_table(tab_conv)
        # criterio de selección simple: si existe asociación significativa entre fixed_var y 'converted' (p_conv < alpha)
        selected = False
        reason = ''
        if isinstance(res_conv.get('p'), float) and res_conv.get('p') < alpha:
            selected = True
            reason = 'p(conversion vs var) < alpha -> asociado con target'
        elif isinstance(res_full.get('p'), float) and res_full.get('p') < alpha:
            selected = True
            reason = 'p(tabla completa) < alpha -> asociado con combinaciones day/time/converted'
        else:
            reason = 'no asociación detectada (p >= alpha)'
        outcomes.append({
            'fixed_var': v,
            'p_full': res_full.get('p'),
            'chi2_full': res_full.get('chi2'),
            'cramers_full': res_full.get('cramers_v'),
            'p_day': res_day.get('p'),
            'p_time': res_time.get('p'),
            'p_conv': res_conv.get('p'),
            'chi2_conv': res_conv.get('chi2'),
            'cramers_conv': res_conv.get('cramers_v'),
            'selected': selected,
            'selection_reason': reason,
            'warning_full': res_full.get('warning'),
            'warning_conv': res_conv.get('warning') if res_conv.get('warning') else ''
        })
    df_out = pd.DataFrame(outcomes)
    if verbose:
        if _HAS_DISPLAY_TOOL:
            display_dataframe_to_user("Resumen pruebas chi2 por variable fija", df_out)
        else:
            print(df_out.to_string(index=False))
    return df_out


# --- Función para mostrar la tabla y resultados para una variable fija escogida ---
def build_and_show_for_var(fixed_var, df=df_calls_voters):
    table_hier, table_flat = build_contingency_table(df, fixed_var)
    title = f"Tabla de contingencia para variable fija: {fixed_var} (índice: day_of_the_week, time_slot, converted)"
    print(title)
    if _HAS_DISPLAY_TOOL:
        display_dataframe_to_user(title, table_flat.reset_index().rename(columns={'index':'combination'}))
    else:
        # mostramos las primeras filas para no saturar la salida
        with pd.option_context('display.max_rows', 200, 'display.max_columns', 200):
            display = table_flat.reset_index().rename(columns={'index':'combination'})
            print(display.head(200).to_string(index=False))
    # correr y mostrar test chi2 para esta variable
    
    res = run_tests_for_fixed_vars(df, fixed_vars_list=[fixed_var], verbose=False)
    if _HAS_DISPLAY_TOOL:
        display_dataframe_to_user("Resultado prueba chi2 (resumen)", res)
    else:
        print("\nResultado prueba chi2 (resumen):\n")
        print(res.to_string(index=False))
    return table_hier, table_flat, res
    

# --- Selector interactivo (si ipywidgets está disponible) ---
def show_interactive_selector(fixed_vars_list=FIXED_VARS, df=df_calls_voters):
    try:
        import ipywidgets as widgets
        from IPython.display import display, clear_output
        dropdown = widgets.Dropdown(options=[v for v in fixed_vars_list if v in df.columns],
                                    description='fixed_var:')
        out = widgets.Output()

        def on_change(change):
            if change['name'] == 'value' and change['new'] is not None:
                with out:
                    clear_output(wait=True)
                    build_and_show_for_var(change['new'], df=df)

        dropdown.observe(on_change)
        display(dropdown, out)
        print("Selector interactivo creado. Selecciona una variable fija en el desplegable.")
    except Exception as e:
        print("ipywidgets no está disponible o ocurrió un error al crear el selector interactivo.")
        print("Puedes llamar a build_and_show_for_var('operator_name') para ver la tabla de otra variable.\nError:", e)


# --- Ejecución: mostramos el selector si el entorno lo permite, y ejecutamos un resumen rápido para todas las fixed vars ---
print("Variables fijas disponibles en el DataFrame (intersección con la lista esperada):")
print([v for v in FIXED_VARS if v in df_calls_voters.columns])

# crear selector interactivo
show_interactive_selector([v for v in FIXED_VARS if v in df_calls_voters.columns], df_calls_voters)

# correr las pruebas para todas las fixed vars y devolver el DataFrame resumen
summary_results = run_tests_for_fixed_vars(df_calls_voters, verbose=True)

# devolvemos objetos útiles (en un notebook esto queda en memoria)
summary_results.head(20)

Variables fijas disponibles en el DataFrame (intersección con la lista esperada):
['genre', 'us_congress', 'operator_name', 'strategy_dial', 'age_range', 'city', 'party', 'ethnic_description', 'st_senate', 'house_district', 'sboe', 'candidate_name']


Dropdown(description='fixed_var:', options=('genre', 'us_congress', 'operator_name', 'strategy_dial', 'age_ran…

Output()

Selector interactivo creado. Selecciona una variable fija en el desplegable.
         fixed_var  p_full     chi2_full  cramers_full         p_day        p_time        p_conv     chi2_conv  cramers_conv  selected                                                           selection_reason                                                                                                                                                            warning_full                                                                                                                                                          warning_conv
             genre     0.0  10584.868875      0.118947  0.000000e+00  0.000000e+00  1.182156e-06     27.296341      0.008542      True                        p(conversion vs var) < alpha -> asociado con target  13 celdas tienen frecuencia esperada < 5. Resultado del chi2 puede no ser fiable. Considera agrupar categorías o usar pruebas exactas/montecarlo para tablas pequeñas.  

,fixed_var,p_full,chi2_full,cramers_full,p_day,p_time,p_conv,chi2_conv,cramers_conv,selected,selection_reason,warning_full,warning_conv
0,genre,0.0,10584.868875,0.118947,0.000000e+00,0.000000e+00,1.182156e-06,27.296341,0.008542,True,p(conversion vs var) < alpha -> asociado con target,13 celdas tienen frecuencia esperada < 5. Resultado del chi2 puede no ser fiable. Considera agrupar categorías o usar pruebas exactas/montecarlo para tablas pequeñas.,
1,us_congress,0.0,218186.675756,0.341551,0.000000e+00,0.000000e+00,2.597031e-176,826.109849,0.046994,True,p(conversion vs var) < alpha -> asociado con target,74 celdas tienen frecuencia esperada < 5. Resultado del chi2 puede no ser fiable. Considera agrupar categorías o usar pruebas exactas/montecarlo para tablas pequeñas.,
2,operator_name,0.0,423312.358335,0.273069,0.000000e+00,0.000000e+00,0.000000e+00,290867.243755,0.816133,True,p(conversion vs var) < alpha -> asociado con target,265 celdas tienen frecuencia esperada < 5. Resultado del chi2 puede no ser fiable. Considera agrupar categorías o usar pruebas exactas/montecarlo para tablas pequeñas.,
3,strategy_dial,0.0,92298.316221,0.325084,7.114688e-174,3.076807e-139,0.000000e+00,90599.861428,0.455488,True,p(conversion vs var) < alpha -> asociado con target,35 celdas tienen frecuencia esperada < 5. Resultado del chi2 puede no ser fiable. Considera agrupar categorías o usar pruebas exactas/montecarlo para tablas pequeñas.,
4,age_range,0.0,61475.402888,0.234054,0.000000e+00,0.000000e+00,2.286004e-95,441.480623,0.034354,True,p(conversion vs var) < alpha -> asociado con target,2 celdas tienen frecuencia esperada < 5. Resultado del chi2 puede no ser fiable. Considera agrupar categorías o usar pruebas exactas/montecarlo para tablas pequeñas.,
5,city,0.0,226398.424340,0.135427,0.000000e+00,0.000000e+00,1.875351e-166,893.891116,0.048884,True,p(conversion vs var) < alpha -> asociado con target,395 celdas tienen frecuencia esperada < 5. Resultado del chi2 puede no ser fiable. Considera agrupar categorías o usar pruebas exactas/montecarlo para tablas pequeñas.,
6,party,0.0,202664.402487,0.278205,0.000000e+00,0.000000e+00,3.796140e-170,807.827408,0.046471,True,p(conversion vs var) < alpha -> asociado con target,110 celdas tienen frecuencia esperada < 5. Resultado del chi2 puede no ser fiable. Considera agrupar categorías o usar pruebas exactas/montecarlo para tablas pequeñas.,
7,ethnic_description,0.0,174626.730679,0.305560,0.000000e+00,0.000000e+00,8.795583e-151,708.079528,0.043508,True,p(conversion vs var) < alpha -> asociado con target,29 celdas tienen frecuencia esperada < 5. Resultado del chi2 puede no ser fiable. Considera agrupar categorías o usar pruebas exactas/montecarlo para tablas pequeñas.,
8,st_senate,0.0,221059.700088,0.213211,0.000000e+00,0.000000e+00,2.062121e-178,873.850358,0.048333,True,p(conversion vs var) < alpha -> asociado con target,133 celdas tienen frecuencia esperada < 5. Resultado del chi2 puede no ser fiable. Considera agrupar categorías o usar pruebas exactas/montecarlo para tablas pequeñas.,2 celdas tienen frecuencia esperada < 5. Resultado del chi2 puede no ser fiable. Considera agrupar categorías o usar pruebas exactas/montecarlo para tablas pequeñas.
9,house_district,0.0,221836.490772,0.186775,0.000000e+00,0.000000e+00,1.910439e-171,858.048796,0.047894,True,p(conversion vs var) < alpha -> asociado con target,269 celdas tienen frecuencia esperada < 5. Resultado del chi2 puede no ser fiable. Considera agrupar categorías o usar pruebas exactas/montecarlo para tablas pequeñas.,4 celdas tienen frecuencia esperada < 5. Resultado del chi2 puede no ser fiable. Considera agrupar categorías o usar pruebas exactas/montecarlo para tablas pequeñas.


# Decisión de selección sobre las variables fijas

**Notas previas importantes:**
Se implementa una prueba de bondad de ajuste que utiliza el estadístico **chi-cuadrado** para determinar si puede inferirse de la muestra de votantes que se llaman una dependencia estadística de variables categóricas. En el caso de uso de este trabajo se utiliza un nivel de significancia del 5% y se implementa además la medida estadística V de Cramér que realiza una correción sobre el estadístico chi_cuadrado eliminando posibles decisiones erróneas a causa del desbalance de clases en **converted** y el tamaño considerable de la muestra de votantes llamados. Dicha medida es función de chi cuadrado, del número total de llamadas lanzadas y del total de categorías distintas para los campos en cuestión. Retorna valores entre 0 y 1 indicando el nivel de asociatividad.  A saber:

Cramér's V: índice de magnitud del efecto (0 = no asociación, 1 = asociación perfecta). Reglas orientativas:

- -~0.0–0.1: negligible / muy pequeño

- ~0.1: pequeño

- ~0.3: moderado

- ~0.5+: fuerte



#### **Criterio de Decisión**:
 - Se revisa la tabla cruzada de tasa de conversión entre la variable potencial explicativa y la combinación entre día de la semana y franja horaria. Las diferencias entre las categorías de la variable explicativa son un buen indicativo de asociatividad. Naturalmente, diferencias del orden de 0.1%-0.5% son insignificantes y sugieren la exclusión de la variable del modelo.
   
 - A continuación se revisan lás métricas **p_conv** y **cramers_conv**. Estas métricas provienen de la tabla cruzada entre las variables **converted** y la **variable explicativa potencial**. Aunque **p_conv** sea pequeño permitiendo concluir que existe asociatividad, **p_value** es frágil cuando el tamaño de la muestra es grande y debe revisarse **Cramer's V** para obtener conclusiones fiables. Obtener por ejemplo **p_value<<1** y cramer>0.3 (al menos existe asociatividad moderada) sugiere dependencia de la variable explicativa abordada y debe incluirse.

**Nota:** Los valores de **p_full** y **cramer_full** son quienes pueden o no sugerir relaciones entre la variable explicativa potencial y la combinación de variables sobre día de la semana y franja horaria. Es decir, pueden sugerir una diferencia significativa en general de la tasa de conversión entre las categorías distintas de la variable expicativa para todas las combinaciones entre día de la semana y franja horaria presentes en la tabla Delta de llamadas.
 
Cada campo se somete a este criterio obteniéndose los siguientes resultados (idealmente transformar en tabla e incluir en la etapa de Análisis descriptivo del informe): 

#### Resultados de criterior de decisión

Se listan los campos y la decisión tomada: 

- **strategy_dial (seleccionado)**: Aunque la proporción de llamadas que usaron *strategy_dial* **Main** es considerablemente menor a la estrategia **None** (63991 llamadas (14.65%) vs 371861 (85.15%)),la tasa de conversión **TC** en **Main** es considerablemente mayor. En **Main** el valor de TC más bajo fue del 15.07% el lunes al mediodía, mientras que el valor más alto ocurre el sábado en la mañana con 37.17%. En el caso de **None** el valor más alto fue de 0.27%. Esto sugiere que áun con menos llamadas, la estrategia de marcado **Main** consiguió una tasa de conversión superior denotando una diferencia de eficiencia signficativa. Los valores **p_conv**<<0.05, **cramer_conv**=0.46, **p_full**<<0.05  y **cramer_full**=0.33, sugieren que la estrategia de marcado influye en la probabilidad de conversión y en cómo varía la conversión según el día y la franja horaria.
  
- **operator_name (seleccionado)**: Destacan en particular **Agent 2** y **Agent 15**, que realizaron 8519 y 1506 llamadas respectivamente, alcanzando los valores más altos de TC. En particular, **Agent 2** logró un **TC** mínimo de 96.43% el miércoles en la mañana. Los demás operadores lograron valores signficativamente menores. **Agent 4** por ejemplo, quien atendió la mayor proporción de llamadas lanzadas, consiguió un valor mínimo y máximo de 0.42% y 18.18% respectivamente. Los valores **p_conv**<<0.05, **cramer_conv**=0,81(cercano a 1), **p_full**<<0 y **cramer_full**=0.27 sugiere dependencia de la dinámica de TC con el operador del Call Center en general y a lo largo de los días y franjas horarias.

- **genre (no seleccionado)**: La revisión de la tabla cruzada de **TC** evidencia en general diferencias insginificantes en **TC** para los géneros **F** y **M** a lo largo de los días y franjas horarias. Los valores **p_conv**<<0.05, **cramer_conv**=0.008542, **p_full**<<0.05 y **cramer_full**=0.12 confirman que aunque la asociación a través de la prueba de bondad de ajuste sea detectable por el gran tamaño de la muestra de votantes llamados, la magnitud de la diferencia en **TC** entre géneros es insignificante. En consecuencia no se seleccionará para el entrenamiento de modelos.

- **age_range (seleccionado condicionalmente)**: En la tabla cruzada de **TC** se percibe una ligera asociatividad de la dinámica de **TC** a lo largo de los días y franjas horarias con la edad categorizada. La pareja de valores **p_conv**<<0.05 y **cramer_conv**=0.034 indican una dependencia insignificante de los grupos etarios y la tasa de conversión. En cambio, la pareja **p_full**<<0.05 y **cramers_full**=0.23 sugieren una dependencia **baja_moderada** en la dinámica temporal de **TC** de días y franjas horarias con la edad. Se **incluye** el campo como variable explicativa fija aunque puede ser suprimida como estrategia para lograr mejores métricas de entrenamiento en los modelos de ML.

- **city (seleccionado condicionalmente**: Las ciudades minoritarias con menos de 2000 llamadas se agrupan en una sola categoría. De manera similar al filtrado realizado sobre **age_range** se percibe una ligera asociatividad entre la evolución temporal por día de la semana y franja horaria de **TC**, y las ciudades. De la prueba de hipótesis para independencia de variables categóricas se obtiene la pareja de valores **p_conv**<<1 y **cramers_conv**=0.049 indicando dependencia débil entre **TC** global y las ciudades. Aún así, **cramers_full**=0.13 sugiere una pequeña dependencia de **TC** por día y franja horaria con el campo city. Se **mantiene** el campo y junto a **age_range** participaría en la estrategia de reducción de campos para buscar aumento de métricas de entrenamiento.

- **us_congress (No seleccionado)**: Una revisión detallada de la tabla cruzada de **TC** revela que entre distritos congresionales de la misma campaña existe una diferencia de **TC** despreciable a lo largo de todas las combinaciones de día y franja horaria. Esto revela que las diferencias en la evolución de la tasa de conversión para distritos congresionales de diversas campañas se deben a factores propios de la campaña de llamadas y no necesariamente a la ciudad. La pareja **p_conv**<<0.05, **cramers_conv**=0.047 revela que a nivel global la probabilidad de conversión de un votante llamado es invariante del distrito congresional. Sin embargo, los resultados **p_full**<<0.05 y **cramers_full**=0.34 muestran una varianza moderada de **TC** respecto al distrito congresional cuando se analiza en términos de día de llamada y franja horaria. Puesto que para congresos de la misma campaña no existe dependencia estadística significativa revelando la existencia de otros factores responsables del valor moderado de **cramers_full**, no se incluye este campo en el entrenamiento.

- **party (No seleccionado)**: Los resultados cualitativos que se reportan aquí guardan similitud con los reportados en **age_range**. Las diferencias de probabilidad de conversión entre grupos no es estadísticamente significativa para establecer a un grado de confianza del 95% una dependencia con la variable objetivo. La pareja de valores **p_conv**<<1 y **cramer_conv**=0.046 justifican la información anterior. De igual forma al evaluar una posible dependencia entre la evolución de **TC** en días y franjas horarias, se reporta un valor de **cramer_full** de 0.028 que indica no asociatividad. No se incluye entonces este campo en el entrenamiento del modelo.

- **ethnic_description (No seleccionado)**: Con una diferencia máxima de 1.01 puntos porcentuales entre los grupos **Ethnic-CVK** y **Ethnic-EVK**, la tasa de conversión entre grupos étnicos es lo suficientemente pequeña para afirmar con una confianza del 95% que en la población total de votantes, las proporciones de contestadores entre etnias son iguales. La prueba de bondad de ajuste lo sustenta al obtener la pareja de valores **p_conv**<<0.05 y **cramers_conv**=0.01728. Así mismo **cramers_full** de 0.05 denota invarianza de la evolución de **TC** en días y franja horaria respecto de los grupos étnicos. No se incluye por tanto en el entrenamiento de modelos de ML.

- **st_senate (No seleccionado)**: Se identifica un patrón que recuerda al encontrado en **us_congress**: la tasa de conversión global **TC** por distrito senatorial evidencia que distritos de la misma campaña presentan valores de **TC** similares. Este resultado sugiere que por campaña, el distrito senatorial no influye en la probabilidad de contestación. Los valores de **p_conv**<<0.05, **cramers_conv**=0.05 apoyan esta conclusión. Además, se calcularon los valores de **p_full** y **cramers_full** por campaña encontrando las tuplas (0.5938, 0.015), (0.99, 0.021), (p_full<<0.05, 0.033) para las tres campañas. Aunque **cramers_conv**=0.22 indica asociatividad baja moderada, no se incluye en el entrenamiento del modelo porque viene a causa de ejecutar un análisis de selección sobre distritos de diferentes campañas.

- **house_district (No seleccionado)**: Como en el caso de **us_congress** y **st_senate**, las tasas de conversión en el conjunto de categorías de **house_district** que pertenecen a la misma campaña son muy similares entre si, tanto a nivel global como por día y franja horaria. Las tres parejas de valores **p_full**, **cramers_full**: (0.99, 0,026), (0.66, 0.015) y (p_full<<0.05, 0.036) confirman que a nivel de campaña no hay suiciente evidencia estadística a un nivel de confianza del 95% para inferir una probable dependencia de **TC** con los distritos senatoriales. Se excluye por tanto del entrenamiento de modelos de ML.

- **sboe**: La tupla de valores **p_full**, **cramers_full** para la única campaña donde este campo no es nulo (0.85, 0.036) evidencia una independencia de la evolución de **TC** respecto del distrito **SBOE**. Se excluye por tanto del entrenamiento de modelos de ML.

- **candidate_name**: Variable que identifica la campaña. A nivel global los valores **p_conv**<<0.05 y **cramers_conv**=0.04 sugieren una independencia entre **TC** y la campaña. En cambio, los valores **p_full**<<0.05 y **cramers_full**=0.51 sugieren una dependencia fuerte de la evolución de **TC** a nivel de día y franja horaria con la campaña. 

## Tabla resumen de selección de variables explicativas

A continuación se presenta una tabla resumen con los campos evaluados, la decisión tomada (seleccionado / no seleccionado / seleccionado condicionalmente) y un resumen conciso de las razones y estadísticas relevantes que motivaron cada decisión.


| Campo | Decisión | Resumen breve |
|---|---:|---|
| **strategy_dial** | **Seleccionado** | `Main` (63,991 llamadas, 14.65%) muestra TC mucho mayor que `None` (371,861, 85.15%); TC_min Main = 15.07% (lun mediodía), TC_max = 37.17% (sáb mañana), None TC_max = 0.27%. Estadísticos: `p_conv << 0.05`, `cramers_conv = 0.46`, `p_full << 0.05`, `cramers_full = 0.33`. La estrategia afecta la probabilidad de conversión y su variación temporal. |
| **operator_name** | **Seleccionado** | `Agent 2` (8,519 llamadas) y `Agent 15` (1,506) registran las TC más altas (`Agent 2` TC_min = 96.43% miércoles mañana). Otros operadores (p.ej. `Agent 4`) muestran TC entre 0.42% y 18.18%. Estadísticos: `p_conv << 0.05`, `cramers_conv = 0.81`, `p_full << 0.05`, `cramers_full = 0.27`. Fuerte dependencia del TC con el operador y su dinámica temporal. |
| **genre** | **No seleccionado** | Aunque `p_conv << 0.05` y `p_full << 0.05` (detectable por gran muestra), los efectos son prácticamente nulos: `cramers_conv = 0.008542`, `cramers_full = 0.12`. Las diferencias de TC entre `F` y `M` son insignificantes en magnitud; se excluye del entrenamiento. |
| **age_range** | **Seleccionado condicionalmente** | Ligera asociatividad temporal: `p_conv << 0.05`, `cramers_conv = 0.034` (muy pequeña); `p_full << 0.05`, `cramers_full = 0.23` (baja–moderada dependencia en la dinámica por día/franja). Incluido como explicativa, pero puede suprimirse si empeora métricas. |
| **city** | **Seleccionado condicionalmente** | Ciudades minoritarias (<2000 llamadas) agrupadas. Estadísticos: `p_conv << 1` (muy pequeño por muestra) y `cramers_conv = 0.049` (débil), `cramers_full = 0.13` (pequeña dependencia temporal). Se mantiene junto con `age_range` para reducción de campos. |
| **us_congress** | **No seleccionado** | A nivel global `p_conv << 0.05`, `cramers_conv = 0.047` indica independencia práctica respecto al distrito. `p_full << 0.05`, `cramers_full = 0.34` muestra varianza moderada por día/franja atribuible a factores de campaña. No se incluye en entrenamiento. |
| **party** | **No seleccionado** | Similar a `age_range`: `p_conv << 1`, `cramers_conv = 0.046` (diferencias pequeñas); `cramers_full = 0.028` sugiere no asociatividad temporal. No se incluye. |
| **ethnic_description** | **No seleccionado** | Diferencia máxima ~1.01 pp entre grupos; `p_conv << 0.05`, `cramers_conv = 0.01728`, `cramers_full = 0.05`. Magnitud demasiado pequeña; se excluye. |
| **st_senate** | **No seleccionado** | Patrón parecido a `us_congress`: TC por distrito senatorial similar dentro de la misma campaña. `p_conv << 0.05`, `cramers_conv = 0.05`. Tripletas por campaña `p_full, cramers_full`: (0.5938, 0.015), (0.99, 0.021), (p_full << 0.05, 0.033). Efecto ligado a campañas, no se incluye. |
| **house_district** | **No seleccionado** | Tasas similares por campaña; tripletas `p_full, cramers_full`: (0.99, 0.026), (0.66, 0.015), (p_full << 0.05, 0.036). No hay evidencia suficiente para incluirlo. |
| **sboe** | **No seleccionado** | Para la campaña con datos: `p_full = 0.85`, `cramers_full = 0.036` → independencia de TC respecto a SBOE. Se excluye. |
| **candidate_name** | **(Variable de campaña)** | `p_conv << 0.05`, `cramers_conv = 0.04` sugiere independencia global débil; pero `p_full << 0.05`, `cramers_full = 0.51` evidencia dependencia fuerte de la evolución temporal de TC con la campaña. Tratar como variable de estrato/agrupación en análisis por campaña. |
